In [1]:
# Full quantum teleportation (with conditional corrections) - Jupyter-ready
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

In [15]:
# Registers
q = QuantumRegister(3, 'q')
m0 = ClassicalRegister(1, 'm0')
m1 = ClassicalRegister(1, 'm1')
out = ClassicalRegister(1, 'out')
qc = QuantumCircuit(q, m0, m1, out)

In [17]:
# Prepare |+> on q0
qc.h(q[0])


In [19]:

# Create Bell pair between q1 and q2
qc.h(q[1])
qc.cx(q[1], q[2])

In [21]:
# Alice's Bell measurement
qc.cx(q[0], q[1])
qc.h(q[0])
qc.measure(q[0], m0)
qc.measure(q[1], m1)

In [23]:

# Classical-controlled corrections using if_test
# Note: (m1, 1) is a tuple (classical register, value)
with qc.if_test((m1, 1)):
    qc.x(q[2])
with qc.if_test((m0, 1)):
    qc.z(q[2])

In [25]:

# Verify by measuring Bob in X-basis (so |+> shows as deterministic 0)
qc.barrier()
qc.h(q[2])
qc.measure(q[2], out)

print(qc.draw(output='text'))

       ┌───┐          ┌───┐┌─┐                                               ░ »
  q_0: ┤ H ├───────■──┤ H ├┤M├───────────────────────────────────────────────░─»
       ├───┤     ┌─┴─┐└┬─┬┘└╥┘                                               ░ »
  q_1: ┤ H ├──■──┤ X ├─┤M├──╫────────────────────────────────────────────────░─»
       └───┘┌─┴─┐└───┘ └╥┘  ║ ┌────── ┌───┐ ───────┐ ┌────── ┌───┐ ───────┐  ░ »
  q_2: ─────┤ X ├───────╫───╫─┤ If-0  ┤ X ├  End-0 ├─┤ If-0  ┤ Z ├  End-0 ├──░─»
            └───┘       ║   ║ └──╥─── └───┘ ───────┘ └──╥─── └───┘ ───────┘  ░ »
                        ║   ║    ║                   ┌──╨──┐                   »
 m0: 1/═════════════════╬═══╩════╬═══════════════════╡ 0x1 ╞═══════════════════»
                        ║   0 ┌──╨──┐                └─────┘                   »
 m1: 1/═════════════════╩═════╡ 0x1 ╞══════════════════════════════════════════»
                        0     └─────┘                                          »
out: 1/═════════════════════

In [31]:
# Run
sim = AerSimulator()
qc_compiled = transpile(qc, sim)
job = sim.run(qc_compiled, shots=1024)
result = job.result()
counts = result.get_counts()
print("Counts (out m1 m0):", counts)
plot_histogram(counts)
plt.show()

Counts (out m1 m0): {'0 0 1': 239, '0 0 0': 261, '0 1 0': 254, '0 1 1': 270}
